<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/10_rate_round2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/rating_200")

Mounted at /content/drive


In [8]:
rating_sheet = pd.read_csv(folder / "rating_sheet_round2.csv", dtype=str, keep_default_na=False)
my_ratings_path = folder / "my_ratings_round2.csv"

if my_ratings_path.exists():
    my_ratings = pd.read_csv(my_ratings_path, dtype=str, keep_default_na=False)
else:
    my_ratings = pd.DataFrame(columns=["file_id", "rating"])

print(len(rating_sheet), "rows in the sheet")
print(len(my_ratings), "rows already rated")

106 rows in the sheet
45 rows already rated


In [9]:
print("Question: same meaning, without effort?  1 = yes, 0 = no")
print()

for i in range(len(rating_sheet)):
    file_id = rating_sheet.loc[i, "file_id"]
    if file_id in list(my_ratings["file_id"]):
        continue

    print("row", i + 1, "of", len(rating_sheet))
    print("correct text:", rating_sheet.loc[i, "correct_text"])
    print("model text:  ", rating_sheet.loc[i, "model_text"])
    answer = input("1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: ")

    if answer == "q":
        break
    if answer not in ["1", "0", "?"]:
        print("wrong key, this row is skipped. Run this cell again to rate it.")
        continue

    my_ratings.loc[len(my_ratings)] = [file_id, answer]
    my_ratings.to_csv(my_ratings_path, index=False)
    print()

Question: same meaning, without effort?  1 = yes, 0 = no

row 46 of 106
correct text: सशस्त्र विद्रोही समूहहरूको
model text:   सशस्त्र विद्रु ही समूहहरूको
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 0

row 47 of 106
correct text: अस्पृश्य जातिहरू छन्
model text:   अस्पृष्य जातिहरू छन्
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 1

row 48 of 106
correct text: धेरै समयसम्म पानी
model text:   धेरै समयोसम्म पानी
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 1

row 49 of 106
correct text: लेक थाना नेटाथाना
model text:   लेग्द थाना नेटा थाना
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 0

row 50 of 106
correct text: सङ्गठितरूपमा लाग्नुभएको थियो
model text:   संगठित रूपमा लाग्नुभएको थियो
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 1

row 51 of 106
correct text: प्रमुख केन्द्र हुनुको साथै
model text:   प्रमुख केन्द्र हुनुको साथ ै
1 = acceptable, 0 = not acceptable, ? = not sure, q = stop: 1

row 52 of 106
correct text

In [10]:
print(len(my_ratings), "rows rated of", len(rating_sheet))
print(my_ratings["rating"].value_counts())

106 rows rated of 106
rating
0    59
1    42
?     5
Name: count, dtype: int64


In [11]:
private = pd.read_csv(folder / "rating_rows_private_round2.csv", dtype={"file_id": str}, keep_default_na=False)
joined = my_ratings.merge(private, on="file_id")

print("unsure rows:", (joined["rating"] == "?").sum())

rated = joined[joined["rating"].isin(["0", "1"])].copy()
rated["rating"] = rated["rating"].astype(int)
print("rated rows (1 or 0):", len(rated))

unsure rows: 5
rated rows (1 or 0): 101


In [12]:
pd.crosstab(rated["group"], rated["rating"])

rating,0,1
group,,
harmless,1,11
needs a human,40,30
real mistake,18,1


In [13]:
rated.groupby("rating")[["wer", "cer"]].mean().round(2)

,wer,cer
rating,,
0,0.68,0.24
1,0.55,0.10


In [14]:
print("WER and rating:", round(rated["rating"].corr(rated["wer"], method="spearman"), 2))
print("CER and rating:", round(rated["rating"].corr(rated["cer"], method="spearman"), 2))

WER and rating: -0.13
CER and rating: -0.48
